In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split, KFold, cross_val_score,TimeSeriesSplit
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.multioutput import MultiOutputRegressor

from catboost import CatBoostRegressor

cat_base = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.03,
    depth=4,
    l2_leaf_reg=3,
    bootstrap_type='Bernoulli',
    subsample=0.8,
    verbose=0,
    random_state=42
)

cat_base.fit(X_train_base, y_train)

y_pred = cat_base.predict(X_test)

cat_r2_base  = r2_score(y_test, y_pred)
cat_mse_base = mean_squared_error(y_test, y_pred)
cat_rmse_base = np.sqrt(cat_mse_base)

def tolerance_accuracy(y_true, y_hat, tol):
    """Return percentage of values within tolerance (works for multi-output)."""
    # y_true/y_hat shape: (n_samples, n_targets)
    return (np.abs(y_true - y_hat) <= tol).mean()

cat_acc_05 = tolerance_accuracy(y_test.values, y_pred, 0.5)
cat_acc_10 = tolerance_accuracy(y_test.values, y_pred, 1.0)
cat_acc_20 = tolerance_accuracy(y_test.values, y_pred, 2.0)
cat_acc_30 = tolerance_accuracy(y_test.values, y_pred, 3.0)

print(f"=== CAT boost Prediction (Test) for Physics augmented ===")
print(f"R²          : {cat_r2_base:.4f}")
print(f"RMSE        : {cat_rmse_base:.3f} MPa")
print(f"MSE         : {cat_mse_base:.4f} MPa")
print("Accuracy (Relative to Target Range 8.0-19.0 MPa):")
print(f"  High Precision (±0.25 MPa): {tolerance_accuracy(y_test.values, y_pred, 0.25) * 100:5.2f}%")
print(f"  Engineering    (±0.50 MPa): {cat_acc_05 * 100:5.2f}%")
print(f"  Acceptable     (±1.00 MPa): {cat_acc_10 * 100:5.2f}%")
print(f"  Upper Bound    (±2.00 MPa): {cat_acc_20 * 100:5.2f}%")

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# 2. Run cross-validation using the_base tscv object
cat_cv_r2 = cross_val_score(cat_base, X_base, y, cv=kf, scoring='r2', n_jobs=-1)

print("\n=== KFolf CV Evaluation Marix for Physics augmented Features ===")
print(f"Mean R² : {cat_cv_r2.mean():.4f}  (±{cat_cv_r2.std():.4f})")
print(f"Scores  : {np.round(cat_cv_r2, 4)}")

fi = pd.DataFrame({
    'feature'   : FEATURES_base,
    'importance': cat_base.feature_importances_
}).sort_values('importance', ascending=False)

print("\n=== Feature Importances (Top 10) ===")
print(fi.head(10).to_string(index=False))